# 简单的 RAG（检索增强生成）系统

## 概述

此代码实现了用于处理和查询 PDF 文档的基本检索增强生成 (RAG) 系统。该系统使用管道对文档进行编码并创建节点。然后，这些节点可用于构建向量索引以检索相关信息。

## 关键组件

1. PDF 处理和文本提取
2. 文本分块以实现易于管理的处理
3. 使用 FAISS 作为支持存储和 OpenAI 嵌入来创建摄取管道
4. 用于查询已处理文档的检索器设置
5. RAG 系统的评估

## 方法详细信息

### 文档预处理

1. 使用 [SimpleDirectoryReader](https://docs.llamaindex.ai/en/stable/module_guides/loading/simpledirectoryreader/) 加载 PDF。
2. 使用具有指定块大小和重叠的 [SentenceSplitter](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/modules/#sentencesplitter) 将文本分割为 [节点/块](https://docs.llamaindex.ai/en/stable/module_guides/loading/documents_and_nodes/)。

### 文本清理

应用自定义转换 `TextCleaner` 来清理文本。这可能会解决 PDF 中的特定格式问题。

### 摄取管道创建

1. OpenAI 嵌入用于创建文本节点的向量表示。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### 搜索器设置

1. 检索器配置为获取给定查询的前 2 个最相关的块。


## 主要特点

1. 模块化设计：摄取过程被封装在单个函数中，以便于重用。
2. 可配置分块：允许调整块大小和重叠。
3. 高效检索：使用FAISS进行快速相似性搜索。
4. 评估：包括评估RAG 系统性能的功能。

## 使用示例

该代码包含一个测试查询：“气候变化的主要原因是什么？”。这演示了如何使用搜索器从已处理的文档中获取相关上下文。

## 评价

该系统包括一个 `evaluate_rag` 函数来评估搜索器的性能，尽管所提供的代码中没有详细说明所使用的具体指标。

## 这种方法的好处

1. 可扩展性：可以通过分块处理大型文档。
2. 灵活性：易于调整块大小和检索结果数量等参数。
3. 高效：利用FAISS在高维空间中进行快速相似性搜索。
4. 与高级 NLP 集成：使用 OpenAI 嵌入来实现最先进的文本表示。

## 结论

这个简单的 RAG 系统为构建更复杂的信息检索和问答系统提供了坚实的基础。通过将文档内容编码为可搜索的支持存储，它可以有效地检索相关信息以响应查询。此方法对于需要快速访问大型文档或文档集合中的特定信息的应用程序特别有用。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install faiss-cpu llama-index python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [1]:
from typing import List
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.schema import BaseNode, TransformComponent
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
import faiss
import os
import sys
from dotenv import load_dotenv

# Original path append replaced for Colab compatibility

EMBED_DIMENSION = 512

# Chunk settings are way different than langchain examples
# Beacuse for the chunk length langchain uses length of the string,
# while llamaindex uses length of the tokens
CHUNK_SIZE = 200
CHUNK_OVERLAP = 50

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Set embeddig model on LlamaIndex global settings
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=EMBED_DIMENSION)

### 阅读文档

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/q_a.json https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/q_a.json


In [ ]:
path = "data/"
node_parser = SimpleDirectoryReader(input_dir=path, required_exts=['.pdf'])
documents = node_parser.load_data()
print(documents[0])

### 支撑存储

In [3]:
# Create FaisVectorStore to store embeddings
faiss_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=faiss_index)

### 文本清理转换

In [4]:
class TextCleaner(TransformComponent):
    """
    Transformation to be used within the ingestion pipeline.
    Cleans clutters from texts.
    """
    def __call__(self, nodes, **kwargs) -> List[BaseNode]:
        
        for node in nodes:
            node.text = node.text.replace('\t', ' ') # Replace tabs with spaces
            node.text = node.text.replace(' \n', ' ') # Replace paragraph seperator with spacaes
            
        return nodes

### 摄取管道

In [5]:
text_splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

# Create a pipeline with defined document transformations and vectorstore
pipeline = IngestionPipeline(
    transformations=[
        TextCleaner(),
        text_splitter,
    ],
    vector_store=vector_store, 
)

In [6]:
# Run pipeline and get generated nodes from the process
nodes = pipeline.run(documents=documents)

### 创建搜索器

In [7]:
vector_store_index = VectorStoreIndex(nodes)
retriever = vector_store_index.as_retriever(similarity_top_k=2)

### 测试搜索器

In [8]:
def show_context(context):
    """
    Display the contents of the provided context list.

    Args:
        context (list): A list of context items to be displayed.

    Prints each context item in the list with a heading indicating its position.
    """
    for i, c in enumerate(context):
        print(f"Context {i+1}:")
        print(c.text)
        print("\n")

In [9]:
test_query = "What is the main cause of climate change?"
context = retriever.retrieve(test_query)
show_context(context)

Context 1:
Chapter 2: Causes of Climate Change  Greenhouse Gases  The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is  essential for life on Earth, as it keeps the planet warm enough to support life. However, human activities have intensified this natural process, leading to a warmer climate.  Fossil Fuels  Burning fossil fuels for energy releases large amounts of CO2.


Context 2:
The Intergovernmental Panel on Climate Change (IPCC) has documented these changes extensively. Ice core samples, tree rings, and ocean sediments provide a historical record that scientists use to understand past climate conditions and predict future trends. The evidence overwhelmingly shows that recent changes are primarily driven by human activities, particularly the emission of greenhou se gases.  Chapter

### 让我们看看它的表现如何：

In [ ]:
import json
from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCaseParams
from evaluation.evalute_rag import create_deep_eval_test_cases

# Set llm model for evaluation of the question and answers 
LLM_MODEL = "gpt-4o"

# Define evaluation metrics
correctness_metric = GEval(
    name="Correctness",
    model=LLM_MODEL,
    evaluation_params=[
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Determine whether the actual output is factually correct based on the expected output."
    ],
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model=LLM_MODEL,
    include_reason=False
)

relevance_metric = ContextualRelevancyMetric(
    threshold=1,
    model=LLM_MODEL,
    include_reason=True
)

def evaluate_rag(query_engine, num_questions: int = 5) -> None:
    """
    Evaluate the RAG system using predefined metrics.

    Args:
        query_engine: Query engine to ask questions and get answers along with retrieved context.
        num_questions (int): Number of questions to evaluate (default: 5).
    """
    
    
    # Load questions and answers from JSON file
    q_a_file_name = "data/q_a.json"
    with open(q_a_file_name, "r", encoding="utf-8") as json_file:
        q_a = json.load(json_file)

    questions = [qa["question"] for qa in q_a][:num_questions]
    ground_truth_answers = [qa["answer"] for qa in q_a][:num_questions]
    generated_answers = []
    retrieved_documents = []

    # Generate answers and retrieve documents for each question
    for question in questions:
        response = query_engine.query(question)
        context = [doc.text for doc in response.source_nodes]
        retrieved_documents.append(context)
        generated_answers.append(response.response)

    # Create test cases and evaluate
    test_cases = create_deep_eval_test_cases(questions, ground_truth_answers, generated_answers, retrieved_documents)
    evaluate(
        test_cases=test_cases,
        metrics=[correctness_metric, faithfulness_metric, relevance_metric]
    )

### 评估结果

In [ ]:
query_engine  = vector_store_index.as_query_engine(similarity_top_k=2)
evaluate_rag(query_engine, num_questions=1)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--simple-rag-with-llamaindex)